# DeepStack

Inteligência Artificial de nível especialista em Heads-Up No-Limit Poker

Raciocinar enquanto joga usando a “intuição” aprimorada por meio de aprendizado profundo para reavaliar sua estratégia a cada decisão .

Tentativa de replicar o que foi feito, adicionando técnicas de outras libs como Libratus e também melhorias possíveis

In [2]:
import pandas as pd

deepdata = pd.read_csv("./data/deepstack_training_data.csv")
deepdata.head()

,player,private_cards,community_cards,betting_rounds,total_pot,action_taken,outcome
0,player1,"['6 of hearts', 'A of spades']","['5 of spades', 'J of spades', 'Q of clubs', '...","[{'fold': 0.7018801043927544, 'call': 0.596648...",7.933804,fold,0.074681
1,player2,"['2 of clubs', '3 of spades']","['5 of spades', 'J of spades', 'Q of clubs', '...","[{'fold': 0.7018801043927544, 'call': 0.596648...",7.933804,fold,0.199752
2,player1,"['8 of spades', '7 of spades']","['5 of clubs', '3 of diamonds', 'Q of clubs', ...","[{'fold': 0.8324133415973884, 'call': 0.680217...",4.510486,call,0.546591
3,player2,"['K of hearts', 'Q of spades']","['5 of clubs', '3 of diamonds', 'Q of clubs', ...","[{'fold': 0.8324133415973884, 'call': 0.680217...",4.510486,raise,-0.311049
4,player1,"['9 of spades', 'J of spades']","['8 of diamonds', '3 of spades', '4 of clubs',...","[{'fold': 0.3413528458146922, 'call': 0.279417...",6.122164,raise,-0.657418


In [12]:
card_encoding = {
        "2": 2, "3": 3, "4": 4, "5": 5, "6": 6, "7": 7, "8": 8, "9": 9,
        "10": 10, "J": 11, "Q": 12, "K": 13, "A": 14,
        "hearts": 1, "diamonds": 2, "clubs": 3, "spades": 4
    }

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

## Função de pré processamento

In [29]:
def preprocess_data(df):
    # Define a codificação para os ranks e naipes
    card_encoding = {
        "2": 2, "3": 3, "4": 4, "5": 5, "6": 6, "7": 7, "8": 8, "9": 9,
        "10": 10, "J": 11, "Q": 12, "K": 13, "A": 14,
        "hearts": 1, "diamonds": 2, "clubs": 3, "spades": 4
    }
    
    # Encode a single card
    def encode_card(card):
        if isinstance(card, list) and all(isinstance(x, int) for x in card):
            return card  # Already encoded
        if not isinstance(card, str):
            raise ValueError(f"Invalid card value: {card}")
        rank, suit = card.split(" of ")
        return [card_encoding[rank], card_encoding[suit]]
    
    # Process private cards
    def process_private_cards(cards):
        if isinstance(cards, list) and all(isinstance(x, int) for x in cards):
            return cards  # Already encoded
        if isinstance(cards, str):
            cards = eval(cards)  # Convert string to list
        if isinstance(cards, list) and len(cards) == 2:
            return encode_card(cards[0]) + encode_card(cards[1])
        else:
            raise ValueError(f"Invalid private cards: {cards}")
    
    df['private_cards'] = df['private_cards'].apply(process_private_cards)
    
    # Process community cards
    def process_community_cards(cards):
        if isinstance(cards, list) and all(isinstance(x, int) for x in cards):
            return cards  # Already encoded
        if isinstance(cards, str):
            cards = eval(cards)  # Convert string to list
        if isinstance(cards, list):
            return sum([encode_card(c) for c in cards], [])  # Flatten encoded cards
        else:
            raise ValueError(f"Invalid community cards: {cards}")
    
    df['community_cards'] = df['community_cards'].apply(process_community_cards)
    
    # Process betting rounds
    def process_betting_rounds(rounds):
        if isinstance(rounds, list) and all(isinstance(r, (float, int)) for r in rounds):
            return rounds  # Already processed
        if isinstance(rounds, str):
            rounds = eval(rounds)  # Convert string to list of dictionaries
        if isinstance(rounds, list) and all(isinstance(r, dict) for r in rounds):
            return [sum(r.values()) for r in rounds]
        else:
            raise ValueError(f"Invalid betting rounds: {rounds}")
    
    df['betting_rounds'] = df['betting_rounds'].apply(process_betting_rounds)
    
    # Combine features into a single list
    df['features'] = df.apply(
        lambda row: row['private_cards'] + row['community_cards'] + row['betting_rounds'] + [row['total_pot']],
        axis=1
    )
    
    # One-hot encode actions
    df = pd.get_dummies(df, columns=['action_taken'])
    
    # Expand features into individual columns
    features = pd.DataFrame(df['features'].to_list(), index=df.index)
    df = pd.concat([features, df.drop(columns=['private_cards', 'community_cards', 'betting_rounds', 'features'])], axis=1)
    
    return df

In [30]:
processed_data = preprocess_data(deepdata)

In [35]:
# split into features and target
X = processed_data.drop(columns=['outcome', 'player'])
y = processed_data['outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Convert boolean columns to float
X_train = X_train.astype({col: 'float32' for col in X_train.select_dtypes('bool').columns})
X_test = X_test.astype({col: 'float32' for col in X_test.select_dtypes('bool').columns})
print(X_train.dtypes)

0                       int64
1                       int64
2                       int64
3                       int64
4                       int64
5                       int64
6                       int64
7                       int64
8                       int64
9                       int64
10                      int64
11                      int64
12                      int64
13                      int64
14                    float64
15                    float64
16                    float64
17                    float64
18                    float64
total_pot             float64
action_taken_call     float32
action_taken_fold     float32
action_taken_raise    float32
dtype: object


## Rede neural

Rede de "intuição"

Linear (Camada Densa)
- Redes densas são ideais para processar dados tabulares ou representações compactas do estado do jogo
- Capturam interações não-lineares entre as entradas, como relação entre cartas e apostas
- Estrutura:
    - Camada de entrada: Aceita um vetor de estado com todas as informações codificadas (ex. cartas, apostas, pote)
    - Camadas ocultas: 2-3 camadas densas com neurônios suficientes para modelar a complexidade do jogo
    - Camada de saída: Um único neurônio para prever o outcome (valor esperado do estado)

ReLU
- A ReLU (Rectified Linear Unit) acelera o aprendizado e reduz problemas de gradiente desaparecendo.
- Adequada para redes densas, pois lida bem com entradas não-normalizadas.

Perceptron Multicamadas (MLP)
- O formato 128 -> 64 -> 1 foi escolhido com base em:
    - 128 neurônios na primeira camada: Suficientes para capturar a alta dimensionalidade das entradas.
    - 64 neurônios na segunda camada: Reduz a dimensionalidade progressivamente.
    - 1 neurônio na camada de saída: Para prever o outcomeoutcome (valor esperado).

## Significado da coluna de outcome

Recompensa Simulada
- Representa o ganho ou perda do jogador em uma rodada, dependendo das cartas, apostas e ações tomadas.
- Geralmente, é um valor numérico que pode ser:
    - Positivo: Se o jogador ganhou fichas nessa rodada.
    - Negativo: Se o jogador perdeu fichas nessa rodada.
    - Zero: Se o jogador empatou ou tomou uma ação sem impacto financeiro direto.

In [32]:
class IntuitionNetwork(nn.Module):
    def __init__(self, input_dim):
        super(IntuitionNetwork, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.fc(x)

In [33]:
# initialize model, loss and optimizer
input_dim = X_train.shape[1]
model = IntuitionNetwork(input_dim=input_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [36]:
# convert data to tensors
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [37]:
print("X_train_tensor shape:", X_train_tensor.shape)
print("y_train_tensor shape:", y_train_tensor.shape)
print("X_test_tensor shape:", X_test_tensor.shape)
print("y_test_tensor shape:", y_test_tensor.shape)

X_train_tensor shape: torch.Size([8000, 23])
y_train_tensor shape: torch.Size([8000, 1])
X_test_tensor shape: torch.Size([2000, 23])
y_test_tensor shape: torch.Size([2000, 1])


In [38]:
# train the model
epochs = 50
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    predictions = model(X_train_tensor)
    loss = criterion(predictions, y_train_tensor)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

Epoch 1/50, Loss: 0.6458
Epoch 2/50, Loss: 0.3543
Epoch 3/50, Loss: 0.4512
Epoch 4/50, Loss: 0.4930
Epoch 5/50, Loss: 0.4279
Epoch 6/50, Loss: 0.3620
Epoch 7/50, Loss: 0.3462
Epoch 8/50, Loss: 0.3702
Epoch 9/50, Loss: 0.3954
Epoch 10/50, Loss: 0.3967
Epoch 11/50, Loss: 0.3771
Epoch 12/50, Loss: 0.3539
Epoch 13/50, Loss: 0.3419
Epoch 14/50, Loss: 0.3453
Epoch 15/50, Loss: 0.3567
Epoch 16/50, Loss: 0.3654
Epoch 17/50, Loss: 0.3649
Epoch 18/50, Loss: 0.3565
Epoch 19/50, Loss: 0.3464
Epoch 20/50, Loss: 0.3406
Epoch 21/50, Loss: 0.3414
Epoch 22/50, Loss: 0.3462
Epoch 23/50, Loss: 0.3507
Epoch 24/50, Loss: 0.3514
Epoch 25/50, Loss: 0.3481
Epoch 26/50, Loss: 0.3433
Epoch 27/50, Loss: 0.3399
Epoch 28/50, Loss: 0.3395
Epoch 29/50, Loss: 0.3415
Epoch 30/50, Loss: 0.3438
Epoch 31/50, Loss: 0.3445
Epoch 32/50, Loss: 0.3431
Epoch 33/50, Loss: 0.3407
Epoch 34/50, Loss: 0.3388
Epoch 35/50, Loss: 0.3385
Epoch 36/50, Loss: 0.3394
Epoch 37/50, Loss: 0.3405
Epoch 38/50, Loss: 0.3408
Epoch 39/50, Loss: 0.

In [39]:
model.eval()
test_predictions = model(X_test_tensor)
test_loss = criterion(test_predictions, y_test_tensor)
print(f"Test Loss: {test_loss.item():.4f}")

Test Loss: 0.3272


In [40]:
print(y_train.min(), y_train.max(), y_train.std())

-0.9999591859864374 0.9996406235746946 0.5812544990985785


### RMSE do dataset = 0.5813

In [42]:
import math
rmse = math.sqrt(test_loss.item())
print(rmse)

0.5719969707972338


### RMSE da predição do modelo = 0.5720

- Modelo está capturando boa parte da variablilidade, mas ainda tem espaço para melhorar